# llama.cpp kernel probe: why is batch-2 decode slow?

On the T4, batch-2 decode was slower per step than batch 4 (17 ms vs 11 ms), and
batch 16-32 ran ~15% slower than the model predicts. This notebook collects evidence
to explain both:

| Experiment | File | Question |
|---|---|---|
| A. Fine batch sweep (1-64) | `probe_fine.csv` | exactly where does step time jump? |
| B. CUDA graphs off | `probe_graphs_off.csv` | is the batch-2 penalty lost graph capture? |
| C. Nsight Systems kernel summary | `kern_<batch>_cuda_gpu_kern_sum.csv` | which kernels run at each batch size? |

Experiment C installs Nsight Systems if needed and is skipped if that fails.

**How to run:** use the same GPU type as the run you want to explain (T4 by default), then
Runtime > Run all (~25 min, mostly the build). Unzip `<gpu>_kernel_probe.zip` into
`measurements/<gpu>_kernel_probe/` and run `python scripts/analyze_probe.py measurements/<gpu>_kernel_probe`.

In [ ]:
# --- configuration -----------------------------------------------------------
LLAMA_CPP_REF = "master"
HF_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
SAVE_TO_DRIVE = False
N_BATCH, N_UBATCH = 2048, 512

NPP, NTG = 512, 64                       # same prompt length as the training rows
FINE_NPL = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 16, 20, 24, 28, 32, 40, 48, 64]
AB_NPL = [1, 2, 3, 4, 8, 16, 32]         # CUDA graphs on vs off
NSYS_NPL = [1, 2, 4, 16, 32]             # kernel summaries
NSYS_NPP, NSYS_NTG = 32, 128             # short prompt so decode kernels dominate the profile

import datetime, json, os, re, shutil, subprocess, sys
import pandas as pd

BIN = "/content/llama.cpp/build/bin"
GGUF = "/content/tinyllama-1.1b-f16.gguf"
KV_BYTES_PER_TOKEN = 2 * 22 * 256 * 2      # TinyLlama: K+V x layers x kv_dim x fp16
WEIGHT_BYTES = 2.2e9


def run(cmd, check=False, env=None, **kw):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True,
                       env={**os.environ, **(env or {})}, **kw)
    if check and r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:], sep="\n")
        raise RuntimeError(f"command failed ({r.returncode})")
    return r


def gpu_slug(name):
    n = name.upper()
    for key, slug in (("T4", "t4"), ("L4", "l4"), ("H100", "h100"), ("A100", "a100")):
        if key in n:
            if key == "H100" and "PCIE" in n:
                return "h100_pcie"
            if key == "A100" and "80GB" in n:
                return "a100_80gb"
            return slug
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def parse_batched_bench(log, **extra):
    rows = []
    for line in log.splitlines():
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        if len(cells) == 10 and cells[0].isdigit():
            pp, tg, b, n_kv = map(int, cells[:4])
            t_pp, s_pp, t_tg, s_tg, t, s = map(float, cells[4:])
            rows.append(dict(pp=pp, tg=tg, b=b, n_kv=n_kv, t_pp=t_pp, s_pp=s_pp,
                             t_tg=t_tg, s_tg=s_tg, t=t, s=s, **extra))
    return pd.DataFrame(rows)


def batched_bench(npp, ntg, npl, env=None, prefix=()):
    """Run llama-batched-bench with flash attention on; returns (CompletedProcess, fa_args)."""
    ctx = max(npl) * (max(npp) + ntg)
    cmd = [*prefix, f"{BIN}/llama-batched-bench", "-m", GGUF, "-ngl", 99, "-c", ctx,
           "-b", N_BATCH, "-ub", N_UBATCH, "-npp", ",".join(map(str, npp)),
           "-ntg", ntg, "-npl", ",".join(map(str, npl))]
    for fa in (["-fa", "on"], ["-fa"], []):           # newer: -fa on|off|auto, older: bare -fa
        r = run(cmd + fa, env=env)
        if r.returncode == 0:
            return r, fa
        print(r.stderr[-1500:])
    return r, None

## 1. Detect the GPU

In [ ]:
q = "name,driver_version,memory.total,power.limit,clocks.max.sm,clocks.max.mem,compute_cap"
info = run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"], check=True).stdout
info = info.strip().splitlines()[0]
print(info)
name, driver, mem, power, sm_max, mem_max, cc = [s.strip() for s in info.split(",")]
SLUG = gpu_slug(name)
MEM_BYTES = float(mem.split()[0]) * 2**20
CUDA_ARCH = cc.replace(".", "")
OUT = f"/content/results_{SLUG}"
os.makedirs(OUT, exist_ok=True)
env = dict(gpu_name=name, driver=driver, memory_total=mem, power_limit=power,
           max_sm_clock=sm_max, max_mem_clock=mem_max, compute_cap=cc, hf_model=HF_MODEL,
           n_batch=N_BATCH, n_ubatch=N_UBATCH,
           timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))
print(f"GPU {name} -> results in {OUT}, building for sm_{CUDA_ARCH}")

# Throttle-reason field was renamed in newer drivers; use whichever this one accepts.
fields = "timestamp,clocks.sm,clocks.mem,power.draw,temperature.gpu,utilization.gpu"
for reason in ("clocks_event_reasons.active", "clocks_throttle_reasons.active", None):
    f = fields + (f",{reason}" if reason else "")
    if run(["nvidia-smi", f"--query-gpu={f}", "--format=csv"]).returncode == 0:
        fields = f
        break
gpu_log = open(f"{OUT}/gpu_log.csv", "w")
gpu_logger = subprocess.Popen(["nvidia-smi", f"--query-gpu={fields}", "--format=csv", "-lms", "500"],
                              stdout=gpu_log)

## 2. Build llama.cpp

In [ ]:
shutil.rmtree("/content/llama.cpp", ignore_errors=True)
run(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_REF,
     "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
env["llama_cpp_ref"] = LLAMA_CPP_REF
env["llama_cpp_commit"] = run(["git", "-C", "/content/llama.cpp", "rev-parse", "HEAD"],
                              check=True).stdout.strip()
print("commit", env["llama_cpp_commit"])

run(["cmake", "-S", "/content/llama.cpp", "-B", "/content/llama.cpp/build",
     "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}",   # this GPU's arch only
     "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"], check=True)
run(["cmake", "--build", "/content/llama.cpp/build", "--config", "Release", "-j", os.cpu_count(),
     "--target", "llama-bench", "llama-batched-bench"], check=True)
assert os.path.exists(f"{BIN}/llama-bench") and os.path.exists(f"{BIN}/llama-batched-bench")

## 3. Model

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(HF_MODEL, allow_patterns=["*.json", "*.safetensors", "tokenizer.model"])
if not any(f.endswith(".safetensors") for f in os.listdir(model_dir)):
    model_dir = snapshot_download(HF_MODEL, allow_patterns=["*.json", "*.bin", "tokenizer.model"])

run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)
convert = [sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", model_dir,
           "--outtype", "f16", "--outfile", GGUF]
r = run(convert)
if r.returncode != 0:   # converter needs a dependency Colab lacks; install its requirements
    print(r.stderr[-2000:])
    run([sys.executable, "-m", "pip", "install", "-q", "-r",
         "/content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt"], check=True)
    run(convert, check=True)
print(f"{os.path.getsize(GGUF) / 1e9:.2f} GB")

## A. Fine batch sweep (default settings)

In [ ]:
r, fa = batched_bench([NPP], NTG, FINE_NPL)
assert fa is not None, "llama-batched-bench failed"
env["fa_arg"] = " ".join(fa) or "(default)"
open(f"{OUT}/probe_fine.log", "w").write(r.stdout + "\n" + r.stderr)
fine = parse_batched_bench(r.stdout + "\n" + r.stderr, graphs="default")
fine["step_ms"] = fine.t_tg / fine.tg * 1e3
fine.to_csv(f"{OUT}/probe_fine.csv", index=False)
fine[["b", "step_ms", "s_tg"]]

## B. Same batches with CUDA graphs disabled

In [ ]:
r, _ = batched_bench([NPP], NTG, AB_NPL, env={"GGML_CUDA_DISABLE_GRAPHS": "1"})
open(f"{OUT}/probe_graphs_off.log", "w").write(r.stdout + "\n" + r.stderr)
off = parse_batched_bench(r.stdout + "\n" + r.stderr, graphs="off")
off["step_ms"] = off.t_tg / off.tg * 1e3
off.to_csv(f"{OUT}/probe_graphs_off.csv", index=False)
off.merge(fine[["b", "step_ms"]], on="b", suffixes=("_graphs_off", "_default"))

## C. Nsight Systems: which kernels run at each batch size

Each batch size is profiled in its own short run (prompt 32, 128 decode steps), so per-step
kernel counts are roughly `instances / 128`.

In [ ]:
def find_nsys():
    path = shutil.which("nsys")
    if path:
        return path
    for root in ("/usr/local/cuda/bin", "/opt/nvidia/nsight-systems"):
        for dirpath, _, files_ in os.walk(root):
            if "nsys" in files_:
                return os.path.join(dirpath, "nsys")
    # Not preinstalled: try NVIDIA's apt repo (package names carry a version).
    run(["apt-get", "update", "-qq"])
    pkgs = run(["apt-cache", "search", "nsight-systems"]).stdout.split("\n")
    names = sorted(p.split()[0] for p in pkgs if p.split() and "nsight-systems" in p.split()[0])
    for pkg in reversed(names):
        if run(["apt-get", "install", "-y", "-qq", pkg]).returncode == 0 and shutil.which("nsys"):
            return shutil.which("nsys")
    return None

NSYS = find_nsys()
env.update(nsys=NSYS, probe_npp=NPP, probe_ntg=NTG, nsys_npp=NSYS_NPP, nsys_ntg=NSYS_NTG)
print("nsys:", NSYS)
if NSYS:
    for b in NSYS_NPL:
        rep = f"/content/prof_{b}"
        r, _ = batched_bench([NSYS_NPP], NSYS_NTG, [b],
                             prefix=(NSYS, "profile", "-o", rep, "--force-overwrite", "true",
                                     "--trace", "cuda", "--sample", "none", "--cpuctxsw", "none"))
        s = run([NSYS, "stats", "--report", "cuda_gpu_kern_sum", "--format", "csv",
                 "--output", f"{OUT}/kern_{b}", "--force-export", "true", rep + ".nsys-rep"])
        print(s.stdout[-500:], s.stderr[-500:])
    print(sorted(f for f in os.listdir(OUT) if f.startswith("kern_")))
else:
    print("Nsight Systems unavailable; experiments A and B still answer most of the question.")

## Package the results

In [ ]:
gpu_logger.terminate()
gpu_log.close()
json.dump(env, open(f"{OUT}/env.json", "w"), indent=2)

g = pd.read_csv(f"{OUT}/gpu_log.csv", skipinitialspace=True)
print(g.describe().loc[["min", "50%", "max"]].to_string())

archive = f"/content/{SLUG}_kernel_probe"
shutil.make_archive(archive, "zip", OUT)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/llm_perf_model", exist_ok=True)
    shutil.copy(archive + ".zip", "/content/drive/MyDrive/llm_perf_model/")
from google.colab import files
files.download(archive + ".zip")